In [ ]:
!apt-get install colmap

In [ ]:

# Импорт необходимых библиотек
import pandas as pd
import os
import shutil
import subprocess
import numpy as np
from scipy.spatial.transform import Rotation as R
import cv2
import sqlite3
import array

# Путь к данным в Kaggle-ноутбуке
data_path = "/kaggle/input/image-matching-challenge-2025"
test_path = os.path.join(data_path, "test")

# Загрузка sample_submission.csv
submission = pd.read_csv(os.path.join(data_path, "sample_submission.csv"))

# Функция для разделения изображений на группы по размерам (для ETs)
def group_images_by_size(dataset, scene, group):
    image_sizes = {}
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            img = cv2.imread(image_path)
            if img is not None:
                size = img.shape[:2]
                size_key = f"{size[0]}x{size[1]}"
                if size_key not in image_sizes:
                    image_sizes[size_key] = []
                image_sizes[size_key].append((idx, row))
    return image_sizes

# Функция для обработки группы изображений
def process_group(dataset, scene, group, indices):
    # Создание временного проектного каталога
    project_dir = f"/kaggle/working/project_{dataset}_{scene}_{str(indices[0])}"
    os.makedirs(os.path.join(project_dir, "images"), exist_ok=True)
    
    # Удаление старой базы данных для перезаписи
    db_path = os.path.join(project_dir, "database.db")
    if os.path.exists(db_path):
        os.remove(db_path)
    
    # Копирование изображений для текущей сцены
    image_list = []
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            shutil.copy(image_path, os.path.join(project_dir, "images", row['image']))
            image_list.append(row['image'])
        else:
            print(f"Warning: Source image not found: {image_path}")
    
    # Пропускаем, если изображений слишком мало
    if len(image_list) < 2:
        print(f"Skipping group with insufficient images: {len(image_list)}")
        shutil.rmtree(project_dir)
        return
    
    # Запуск COLMAP: извлечение признаков с помощью встроенного SIFT
    try:
        result = subprocess.run([
            "colmap", "feature_extractor",
            "--database_path", os.path.join(project_dir, "database.db"),
            "--image_path", os.path.join(project_dir, "images"),
            "--SiftExtraction.use_gpu", "0",
            "--SiftExtraction.max_image_size", "2000"
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print("Feature extractor output:", result.stdout)
        print("Feature extractor errors:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error in feature_extractor: {e.stderr}")
        shutil.rmtree(project_dir)
        return
    
    # Отладка: анализ числа признаков для каждого изображения
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Получаем данные из таблиц images и keypoints
    cursor.execute("SELECT image_id, name FROM images")
    image_data = dict(cursor.fetchall())
    cursor.execute("SELECT image_id, data FROM keypoints")
    keypoints_data = cursor.fetchall()
    
    filtered_images = []
    for image_id, keypoints_data in keypoints_data:
        # Преобразуем бинарные данные в массив и считаем количество точек
        keypoints_array = array.array('f', keypoints_data)
        num_keypoints = len(keypoints_array) // 4  # Каждой точке соответствует 4 значения (x, y, scale, orientation)
        image_name = image_data.get(image_id)
        if num_keypoints < 100:
            print(f"Skipping image {image_name} with only {num_keypoints} keypoints")
            cursor.execute("DELETE FROM images WHERE image_id = ?", (image_id,))
            cursor.execute("DELETE FROM keypoints WHERE image_id = ?", (image_id,))
            cursor.execute("DELETE FROM descriptors WHERE image_id = ?", (image_id,))
        else:
            filtered_images.append(image_name)
    
    conn.commit()
    conn.close()
    
    # Пропускаем, если после фильтрации осталось меньше 2 изображений
    if len(filtered_images) < 2:
        print(f"Skipping group after filtering: {len(filtered_images)} images remain")
        shutil.rmtree(project_dir)
        return
    
    # Запуск COLMAP: исчерпывающее сопоставление признаков
    try:
        result = subprocess.run([
            "colmap", "exhaustive_matcher",
            "--database_path", os.path.join(project_dir, "database.db"),
            "--SiftMatching.use_gpu", "0",
            "--SiftMatching.min_num_inliers", "3",
            "--SiftMatching.max_distance", "1.0",
            "--SiftMatching.max_error", "8.0",
            "--SiftMatching.max_num_matches", "10000",
            "--SiftMatching.min_inlier_ratio", "0.1"
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print("Matcher output:", result.stdout)
        print("Matcher errors:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error in exhaustive_matcher: {e.stderr}")
        shutil.rmtree(project_dir)
        return
    
    # Отладка: изучение структуры базы данных
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM matches")
    match_count = cursor.fetchone()[0]
    print(f"Number of matches found: {match_count}")
    
    cursor.execute("SELECT COUNT(DISTINCT pair_id) FROM matches")
    unique_pairs = cursor.fetchone()[0]
    print(f"Number of unique image pairs with matches: {unique_pairs}")
    
    cursor.execute("SELECT COUNT(*) FROM two_view_geometries")
    two_view_count = cursor.fetchone()[0]
    print(f"Number of verified geometries: {two_view_count}")
    
    conn.close()
    
    # Создание директории для sparse-модели
    os.makedirs(os.path.join(project_dir, "sparse"), exist_ok=True)
    
    # Запуск COLMAP: восстановление сцены с ослабленной фильтрацией
    try:
        result = subprocess.run([
            "colmap", "mapper",
            "--database_path", os.path.join(project_dir, "database.db"),
            "--image_path", os.path.join(project_dir, "images"),
            "--output_path", os.path.join(project_dir, "sparse"),
            "--Mapper.min_num_matches", "3",
            "--Mapper.ignore_watermarks", "1",
            "--Mapper.init_min_num_inliers", "5"
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print("Mapper output:", result.stdout)
        print("Mapper errors:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error in mapper: {e.stderr}")
        shutil.rmtree(project_dir)
        return
    
    # Преобразование модели в текстовый формат
    try:
        result = subprocess.run([
            "colmap", "model_converter",
            "--input_path", os.path.join(project_dir, "sparse", "0"),
            "--output_path", os.path.join(project_dir, "sparse", "0"),
            "--output_type", "TXT"
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except subprocess.CalledProcessError as e:
        print(f"Error in model_converter: {e.stderr}")
        return
    
    # Парсинг images.txt для извлечения поз камер
    images_txt_path = os.path.join(project_dir, "sparse", "0", "images.txt")
    if os.path.exists(images_txt_path):
        with open(images_txt_path, "r") as f:
            lines = f.readlines()
        image_data = {}
        i = 0
        while i < len(lines):
            if lines[i].startswith('#') or lines[i].strip() == '':
                i += 1
                continue
            parts = lines[i].strip().split()
            image_id, qw, qx, qy, qz, tx, ty, tz, camera_id, name = parts
            name = name.replace("images/", "")
            quat = [float(qw), float(qx), float(qy), float(qz)]
            trans = [float(tx), float(ty), float(tz)]
            rot_mat = R.from_quat(quat).as_matrix().flatten()
            rot_flat = ";".join(map(str, rot_mat))
            trans_flat = ";".join(map(str, trans))
            image_data[name] = (rot_flat, trans_flat)
            i += 2
    
        # Заполнение submission
        for idx in indices:
            row = submission.iloc[idx]
            image_name = row['image']
            if image_name in image_data:
                rot, trans = image_data[image_name]
                submission.at[idx, 'rotation_matrix'] = rot
                submission.at[idx, 'translation_vector'] = trans
            else:
                print(f"Warning: {image_name} not found in reconstruction.")
    
    # Удаление временного каталога
    shutil.rmtree(project_dir)

# Группировка изображений по dataset и scene
groups = submission.groupby(['dataset', 'scene'])

# Установка headless-режима для COLMAP
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# Обработка каждой сцены
for (dataset, scene), group in groups:
    # Пропускаем, если директория dataset не существует
    dataset_path = os.path.join(test_path, dataset)
    if not os.path.exists(dataset_path):
        print(f"Skipping non-existent dataset: {dataset}")
        continue
    
    # Для набора ETs разделяем изображения по размерам
    if dataset == 'ETs':
        image_groups = group_images_by_size(dataset, scene, group)
        for size_key, size_group in image_groups.items():
            print(f"Processing ETs group with size {size_key}")
            # Создаем временную группу для обработки
            temp_group = pd.DataFrame([row for _, row in size_group])
            temp_indices = [idx for idx, _ in size_group]
            process_group(dataset, scene, temp_group, temp_indices)
    else:
        # Для других наборов (например, stairs) обрабатываем как обычно
        process_group(dataset, scene, group, group.index)

# Сохранение обновленного submission.csv
submission.to_csv('submission.csv', index=False)